# NextRun — train the court and ball/rim models (Colab, GPU)

**Runtime → Change runtime type → T4 GPU** (or better), add the secret **`ROBOFLOW_API_KEY`** (key icon, left),
then **Runtime → Run all**. Allow Google Drive access when asked.

**Safe to re-run.** Everything that takes time is kept in Google Drive (`MyDrive/nextrun_colab/`): the built
datasets, every training checkpoint and the reports. If the session disconnects, reconnect and *Run all* again:
finished steps are skipped and unfinished training resumes from its last epoch.

| Model | Architecture | Training data | Held-out checks |
|---|---|---|---|
| Court (24 keypoints) | YOLO26m-pose, raced against a fine-tune of the current court model | NBA broadcast · TrackID3x3 outdoor 3x3 + indoor gym (auto-labelled through their calibrations) | test split per source · **TrackID3x3 drone court** (never trained on) |
| Ball + rim | YOLO26m @ 1024 px | 6 Roboflow + 3 Hugging Face sets (~10.8k frames) | test split per set · ball / rim found on our own footage |

Disk / memory: raw videos are deleted after frames are extracted (peak ~30 GB of ~110 GB); data loaders use
2 workers (RAM < 12 GB); batch size adapts to the GPU (AutoBatch).

In [ ]:
import os, json, shutil, subprocess, glob
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/nextrun_colab"
os.makedirs(f"{DRIVE}/results", exist_ok=True); os.makedirs(f"{DRIVE}/datasets", exist_ok=True)

if not os.path.isdir("/content/nextrun"):
    !git clone -q -b model_merge https://github.com/aisutd/nextrun.git /content/nextrun
%cd /content/nextrun
!git pull -q
!pip install -q -e ".[detect,train]" gdown
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
# results/ lives in Drive: checkpoints and reports survive a disconnect
if not os.path.islink("results"):
    !cp -rn results/. {DRIVE}/results/ && rm -rf results && ln -s {DRIVE}/results results
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; df -h /content | tail -1

def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True)
    if r.returncode: raise RuntimeError(f"failed ({r.returncode}): {cmd}")

def dataset(name, build):
    """Restore data/<name> from Drive, or build it once and keep a copy there."""
    tar = f"{DRIVE}/datasets/{name}.tar"
    if os.path.isdir(f"data/{name}/train"):
        return
    if os.path.exists(tar):
        sh(f"mkdir -p data && tar -xf {tar} -C data"); return
    build()
    sh(f"tar -cf {tar}.part -C data {name} && mv {tar}.part {tar}")

def train(task, name, args):
    """Train once; after a disconnect, resume from the last epoch; skip if finished."""
    run = f"results/runs/{task}/{name}"
    if os.path.exists(f"{run}/DONE"):
        print(f"{name}: already trained"); return
    if os.path.exists(f"{run}/weights/last.pt"):
        try:
            sh(f"python src/scripts/train.py {task} --name {name} --resume")
        except RuntimeError:   # finished just before a disconnect: nothing left to resume
            if not os.path.exists(f"{run}/weights/best.pt"):
                raise
    else:
        sh(f"python src/scripts/train.py {task} --name {name} --workers 2 --no-install {args}")
    open(f"{run}/DONE", "w").close()

def once(path, cmd):
    """Run an eval command once; its output is kept in results/."""
    if not os.path.exists(path):
        sh(f"{cmd} > {path}.part && mv {path}.part {path}")
    print(open(path).read())

## 1. Court model
Sources: NBA broadcast (850 frames, Hugging Face, CC BY 4.0) · TrackID3x3 outdoor (12 recordings, one fixed
camera, re-registered per recording) and indoor gym (42 clips) · TrackID3x3 drone court as a **test-only**
source. Add Rishabh's 24-point sets to `COURT_ROBOFLOW` (`workspace/project/version`) to also cover and measure
high-school gyms.

In [ ]:
COURT_ROBOFLOW = [
    # "workspace/project/3",   # Rishabh: corner view, high-school gyms
]
T = "data/raw/trackid3x3/videos"
OUTDOOR = {"IMG_0104": "1B4WgkrIrPaIe-kIRd5Kq8XouUmezolL_", "IMG_0105": "1JmRa6HJp-06pyxHp11G0YOb05tf1wtcC",
           "IMG_0106": "1TGBRoqUcUqZUL33tI1up9OobLtOCQNhw", "IMG_0107": "12Ge8yz1W2BKnrxpX-TM1_ZttArVLnMc0",
           "IMG_0108": "13j5W53tJqPYpTKSFI_zyi3LMqZVy-EFH", "IMG_0109": "134s82zCspw8zdSYBa21__fHLnIXSialZ",
           "IMG_0110": "1H93_doyUuuXQg1SWR-g5FQRXjjUf5bSu", "IMG_0111": "1FJeSP6gSITTjyA6IPJWKlfVDu4RlOWfF",
           "IMG_0112": "1Oc1YlDw4eUfXV-iga4Z_dbkGhVZ-67Pp", "IMG_0113": "1gL63ham7IZM6SLGVTCu-wcJwc3weJ9eL",
           "IMG_0114": "13sIrq7X5xn2Fp-_kQS_mRpI7NoH5PD6M", "IMG_0115": "1e3-gRZ4luCu_2XwiZpuHQpGUHqtostB1"}
FOOTAGE = {"IMG_0105": OUTDOOR["IMG_0105"], "IMG_0110": OUTDOOR["IMG_0110"]}   # kept for the ball footage check

def get_videos(names):
    os.makedirs(f"{T}/outdoor", exist_ok=True); os.makedirs(f"{T}/drone", exist_ok=True)
    for name, fid in names.items():
        if not os.path.exists(f"{T}/outdoor/{name}.MOV"):
            sh(f"gdown -q {fid} -O {T}/outdoor/{name}.MOV")
    if not os.path.isdir(f"{T}/indoor"):
        sh(f"gdown -q --folder 1q0L_Mmf5cwb-Yv51NTjgQcF12Is0fYCB -O {T}/indoor")

def build_court():
    get_videos(OUTDOOR)
    if not os.path.exists(f"{T}/drone/1-1.mov"):
        sh(f"gdown -q 1Xd5dcy3N5SyvdQ887lownjvL38fLl5JD -O {T}/drone/1-1.mov")
    rf = ("--roboflow " + " ".join(COURT_ROBOFLOW)) if COURT_ROBOFLOW else ""
    sh(f"python src/scripts/build_court_dataset.py --out data/court24 --nba-hf {rf} "
       f"--venue trackid3x3_outdoor '{T}/outdoor/IMG_0114.MOV,{T}/outdoor/*.MOV' --venue-ref-frame trackid3x3_outdoor 900 "
       f"--venue trackid3x3_indoor '{T}/indoor/*.mp4' "
       f"--venue trackid3x3_drone {T}/drone/1-1.mov --venue-ref-frame trackid3x3_drone 300 --test-venue trackid3x3_drone "
       f"--every 300")
    for v in glob.glob(f"{T}/outdoor/*.MOV") + glob.glob(f"{T}/drone/*"):   # frames are extracted; free the disk
        if os.path.basename(v)[:-4] not in FOOTAGE:
            os.remove(v)

dataset("court24", build_court)
!cat data/court24/sources.json; df -h /content | tail -1
once("results/court_current_test.txt", "python src/scripts/eval_court.py --data data/court24 --split test")

In [ ]:
# candidate A: YOLO26m-pose from COCO-pretrained weights, 1280 px, capped at 1.75 h
train("court", "court_yolo26m", "--data data/court24/data.yaml --model yolo26m-pose.pt --imgsz 1280 --epochs 200 --hours 1.75")
once("results/court_yolo26m_test.txt", "python src/scripts/eval_court.py --data data/court24 --split test --weights results/runs/court/court_yolo26m/weights/best.pt")
# candidate B: fine-tune the current court model (keeps its high-school-gym training), capped at 0.75 h
train("court", "court_finetune", "--data data/court24/data.yaml --model models/court_model.pt --imgsz 1280 --epochs 100 --hours 0.75")
once("results/court_finetune_test.txt", "python src/scripts/eval_court.py --data data/court24 --split test --weights results/runs/court/court_finetune/weights/best.pt")

In [ ]:
# pick: a candidate must find the court at least as often and be at least as accurate as the current model on
# every test source (drone court included); among those, the lowest mean median error wins
load = lambda p: json.loads(open(p).read().splitlines()[-1])
cur = load("results/court_current_test.txt")
cands = {"court_yolo26m": load("results/court_yolo26m_test.txt"), "court_finetune": load("results/court_finetune_test.txt")}
nan = lambda v: v != v
def ok(new):
    return all(new[s]["found_rate"] >= cur[s]["found_rate"] - 0.02 and
               (nan(cur[s]["err_ft_median"]) or (not nan(new[s]["err_ft_median"]) and new[s]["err_ft_median"] <= cur[s]["err_ft_median"]))
               for s in cur)
score = lambda r: sum(r[s]["err_ft_median"] for s in r if not nan(r[s]["err_ft_median"])) / len(r)
for n, r in [("current", cur)] + list(cands.items()):
    print(f"{n:16s} " + "  ".join(f"{s}: {r[s]['found']}/{r[s]['images']} {r[s]['err_ft_median']} ft" for s in r))
good = [n for n, r in cands.items() if ok(r)]
best = min(good, key=lambda n: score(cands[n])) if good else None
print("INSTALL", best if best else "nothing - current court model stays")
if best:
    shutil.copyfile("models/court_model.pt", "models/court_model.prev.pt")
    shutil.copyfile(f"results/runs/court/{best}/weights/best.pt", "models/court_model.pt")
    shutil.copyfile(f"results/runs/court/{best}/weights/best.pt", "results/court_model_installed.pt")
json.dump({"installed": best}, open("results/court_choice.json", "w"))

## 2. Ball + rim model
6 Roboflow sets + 3 Hugging Face mirrors that label both ball and rim (rim boxes = rim + net). Add your own
labelled frames (Roboflow export) to `BALL_ROBOFLOW` — frames from **your** cameras help most.

In [ ]:
BALL_ROBOFLOW = ["suggested"]   # + "your-workspace/your-project/1"
dataset("ball_rim", lambda: sh(f"python src/scripts/build_ball_rim_dataset.py --out data/ball_rim --hf suggested --roboflow {' '.join(BALL_ROBOFLOW)}"))
!rm -rf data/raw/roboflow data/raw/hf; df -h /content | tail -1
once("results/ball_current_test.txt", "python src/scripts/eval_ball_rim.py --data data/ball_rim --split test")

In [ ]:
# YOLO26m @ 1024 px (YOLO26 targets small objects; balls are tiny in wide shots), capped at 4 h
train("ball", "ball_yolo26m", "--data data/ball_rim/data.yaml --model yolo26m.pt --imgsz 1024 --epochs 120 --hours 4")
once("results/ball_yolo26m_test.txt", "python src/scripts/eval_ball_rim.py --data data/ball_rim --split test --weights results/runs/ball/ball_yolo26m/weights/best.pt")
# our own footage (labelled sets are mostly broadcast): ball / rim found per sampled frame
get_videos(FOOTAGE)
once("results/ball_footage.txt", f"python src/scripts/check_ball_footage.py {T}/outdoor/IMG_0105.MOV {T}/outdoor/IMG_0110.MOV "
     f"{T}/indoor/basket_S3T2_pre.mp4 {T}/indoor/basket_S5T4_post.mp4 "
     f"--weights models/ball_hoop_model.pt results/runs/ball/ball_yolo26m/weights/best.pt --save results/ball_disagreements")

In [ ]:
cur, new = load("results/ball_current_test.txt"), load("results/ball_yolo26m_test.txt")
foot = load("results/ball_footage.txt")
old_f, new_f = foot["models/ball_hoop_model.pt"], foot["results/runs/ball/ball_yolo26m/weights/best.pt"]
for s in cur:
    for c in ("ball", "rim"):
        if c in cur[s] and c in new.get(s, {}):
            print(f"{s:40s} {c:4s} recall {cur[s][c]['R']} -> {new[s][c]['R']}   mAP50 {cur[s][c]['mAP50']} -> {new[s][c]['mAP50']}")
mean = lambda d, c: sum(d[s][c]["mAP50"] for s in d if c in d[s]) / max(1, sum(c in d[s] for s in d))
print(f"mean mAP50 ball {mean(cur, 'ball'):.3f} -> {mean(new, 'ball'):.3f}   rim {mean(cur, 'rim'):.3f} -> {mean(new, 'rim'):.3f}")
print(f"own footage: ball found {old_f['ball_found']} -> {new_f['ball_found']} of {new_f['frames']}   rim {old_f['rim_found']} -> {new_f['rim_found']}")
install = (mean(new, "ball") > mean(cur, "ball") and mean(new, "rim") >= mean(cur, "rim") - 0.01
           and new_f["ball_found"] >= old_f["ball_found"] and new_f["rim_found"] >= old_f["rim_found"])
print("INSTALL ball_yolo26m" if install else "NOT better on both - current ball model stays (see results/ball_disagreements)")
if install:
    shutil.copyfile("models/ball_hoop_model.pt", "models/ball_hoop_model.prev.pt")
    shutil.copyfile("results/runs/ball/ball_yolo26m/weights/best.pt", "models/ball_hoop_model.pt")
    shutil.copyfile("results/runs/ball/ball_yolo26m/weights/best.pt", "results/ball_model_installed.pt")
json.dump({"installed": "ball_yolo26m" if install else None}, open("results/ball_choice.json", "w"))

## 3. Results
Everything is already in Google Drive (`MyDrive/nextrun_colab/results`). This also downloads a zip of the
reports and the installed weights (training runs excluded).

In [ ]:
!cd results && zip -qr /content/nextrun_colab_results.zip . -x "runs/*" "ball_disagreements/*"
open("results/ALL_DONE", "w").close()
from google.colab import files
files.download("/content/nextrun_colab_results.zip")